# Getting Started with Tentris: A Unified Notebook

Welcome! This notebook provides a simple, interactive way to use Tentris. It **automatically detects your operating system** (Linux, Windows, or macOS) and uses the best method to run.

-   **On Linux:** It uses the simple and efficient **in-memory** store.
-   **On Windows & macOS:** It will guide you through running a **local server process**.

## 🛑 IMPORTANT: One-Time Setup (for Windows & macOS Users)

If you are on Windows or macOS, you must do this once:

1.  **Download Tentris:** Go to the [Tentris Releases page](https://github.com/tentris/tentris/releases) and download the correct version for your operating system.
2.  **Extract the file:** Unzip the downloaded file.
3.  **Place it here:** Move the executable file (`tentris.exe` or `tentris`) into the **exact same folder as this notebook file**.

### Step 1: Install Dependencies

Run the following cell once to install all the necessary Python libraries for all operating systems.

In [1]:
!pip install tentris rdflib pandas requests

### Step 2: Imports and Initial Setup

This cell imports all necessary libraries.

In [2]:
import sys
import os
import subprocess
import time
import requests
import pandas as pd
import rdflib
import tentris
from IPython.display import display, Markdown

# Global variables to hold our database connection and server process
graph = None
server_process = None

### Step 3: Initialize Tentris (OS-Aware)

This is the core logic. The cell checks your OS and runs the appropriate setup code.

In [6]:
if sys.platform == "linux":
    # --- LINUX PATH ---
    display(Markdown("🐧 **Linux detected.** Using the simple in-memory store."))
    # graph # <-- FIX: Declare that we are modifying the global 'graph' variable
    try:
        graph = rdflib.Graph(store="Tentris")
        display(Markdown("✅ In-memory database created successfully!"))
    except Exception as e:
        display(Markdown(f"❌ **Error creating in-memory store:** {e}"))
else:
    # --- WINDOWS / MACOS PATH ---
    display(Markdown("💻 **Windows/macOS detected.** Starting local Tentris server."))
    #global graph # <-- FIX: Declare that we are modifying the global 'graph' variable
    
    TENTRIS_BINARY = "./tentris.exe" if sys.platform == 'win32' else "./tentris"
    DATASTORE_PATH = "./tentris_notebook_data"


 

    def start_server():
        global server_process
        if os.path.exists(DATASTORE_PATH):
            display(Markdown(f"🧹 Deleting old datastore..."))
            if sys.platform == 'win32': os.system(f'rmdir /s /q "{DATASTORE_PATH}"')
            else: os.system(f"rm -rf '{DATASTORE_PATH}'")
        
        if not os.path.exists(TENTRIS_BINARY):
            display(Markdown(f"❌ **Error:** Cannot find `{TENTRIS_BINARY}`. Please complete the one-time setup steps at the top."))
            return
            
        display(Markdown(f"🚀 Starting server from `{TENTRIS_BINARY}`..."))
        command = [TENTRIS_BINARY, "serve", "--datastore-path", DATASTORE_PATH]
        server_process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        
        display(Markdown("...waiting 3 seconds for initialization..."))
        time.sleep(3)
        
        try:
            requests.get("http://localhost:9080/status", timeout=2)
            display(Markdown("✅ **Server is up and running!**"))
        except requests.ConnectionError:
            display(Markdown("❌ **Server failed to start.** Error logs:"))
            print(server_process.stderr.read().decode())
            server_process = None
            
    start_server()
    
    if server_process:
        graph = rdflib.Graph(store="TentrisHTTP")
        display(Markdown("✅ Connected to local Tentris server."))

print(graph)

🐧 **Linux detected.** Using the simple in-memory store.

✅ In-memory database created successfully!

[a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'TentrisStore']].


### Step 4: Load Data

This cell will work no matter which setup was used above, as long as the `graph` object was created successfully.

In [9]:
display(Markdown("⬇️ Loading Mona Lisa knowledge graph..."))
try:
    graph.update("LOAD <https://files.tentris.io/mona-lisa.ttl>")
    count_result = graph.query("SELECT (COUNT(*) AS ?c) WHERE { ?s ?p ?o }")
    
    # Iterate through the result to get the single row
    for row in count_result:
        triple_count = row.c
        break # We only expect one row
    
    display(Markdown(f"✔️ Data loaded successfully. Total triples: **{triple_count}**"))
except Exception as e:
    display(Markdown(f"❌ **Error loading data:** {e}"))


⬇️ Loading Mona Lisa knowledge graph...

✔️ Data loaded successfully. Total triples: **14**

### Step 5: Run SPARQL Queries

Now for the fun part! Let's find out the name of the Mona Lisa's author.

In [10]:
if graph:
    query_str = """
       PREFIX foaf: <http://xmlns.com/foaf/0.1/>
       PREFIX dbr: <http://dbpedia.org/resource/>
       PREFIX dbo: <http://dbpedia.org/ontology/>
    
       SELECT ?name WHERE {
          dbr:Mona_Lisa dbo:author ?person .
          ?person foaf:name ?name .
       }
    """
    
    display(Markdown("### Query Results:"))
    for row in graph.query(query_str):
        print(f"🎨 The author is: {row.name}")

### Query Results:

🎨 The author is: Leonardo da Vinci


### Step 6: Shut Down

This is a **very important step**. This cell will correctly shut down the database, whether it was an in-memory instance or a background server.

In [11]:
if graph:
    # If server_process is not None, it means we are on Win/macOS
    if server_process:
        display(Markdown("🛑 Stopping Tentris server..."))
        server_process.terminate()
        server_process.wait()
        server_process = None
        display(Markdown("✅ Server shut down successfully."))
    else:
        # Otherwise, we are on Linux
        display(Markdown("🧠 Closing in-memory database..."))
        graph.close()
        display(Markdown("✅ In-memory database shut down and cleared."))
    
    graph = None
    display(Markdown("All clean! You can re-run Step 3 to start again."))
else:
    display(Markdown("🤷 Nothing to shut down."))

🧠 Closing in-memory database...

✅ In-memory database shut down and cleared.

All clean! You can re-run Step 3 to start again.